## Imports

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mygene
import shap
import xgboost as xgb
import joblib
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings('ignore')

os.makedirs('../figures', exist_ok=True)

RANDOM_STATE = 42
N_HVG       = 3000   # HVG count
TARGET_MIN  = 1500   # Minimal cross
TARGET_MAX  = 2000   # Maximum cross
ELIM_STEP   = 5      # Number of features eliminated per iterations
ELIM_MIN    = 150    # Minimal feature count 
ELIM_MAX    = 250    # Target feature count

ModuleNotFoundError: No module named 'mygene'

## Data loading

In [ ]:
# TCGA star counts
BULK_PATH = '../data/TCGA-LUAD.star_counts.tsv.gz'
META_PATH = '../data/41586_2014_BFnature13385_MOESM21_ESM.xlsx'

tcga = pd.read_csv(BULK_PATH, sep='\t', index_col=0)
tcga.index = tcga.index.str.split('.').str[0]

tcga = (2 ** tcga) - 1 # log2(count+1) → counts

tumor_cols  = tcga.columns[tcga.columns.str.endswith('-01A')]
normal_cols = tcga.columns[tcga.columns.str.endswith('-11A')]

tcga_tumor  = tcga[tumor_cols].copy()
tcga_normal = tcga[normal_cols].copy()

tcga_tumor.columns = tcga_tumor.columns.str[:12]
tcga_tumor = tcga_tumor.loc[:, ~tcga_tumor.columns.duplicated()]

print(f'Tumor:  {tcga_tumor.shape}')
print(f'Normal: {tcga_normal.shape}')

Tumor:  (60660, 513)
Normal: (60660, 58)


In [ ]:
meta = pd.read_excel(META_PATH, sheet_name=6, header=4)
meta = meta[['Tumor ID', 'expression_subtype']].dropna()

common_ids = list(set(tcga_tumor.columns) & set(meta['Tumor ID']))
tcga_train = tcga_tumor[common_ids].T  # samples x genes
labels = meta.set_index('Tumor ID').loc[common_ids, 'expression_subtype']

tcga_train = tcga_train.div(tcga_train.sum(axis=1), axis=0) * 1e6
tcga_train = np.log2(tcga_train + 1)

tcga_normal_t = tcga_normal.T
tcga_normal_t = tcga_normal_t.div(tcga_normal_t.sum(axis=1), axis=0) * 1e6
tcga_normal_t = np.log2(tcga_normal_t + 1)
# BINARY LABELS
labels_binary = labels.map(lambda x: 'TRU' if x == 'TRU' else 'NON-TRU')
le = LabelEncoder()
y = le.fit_transform(labels_binary.values)

print(f'Train tumor: {tcga_train.shape}')
print(f'Normal:      {tcga_normal_t.shape}')
print(f'Classes: {dict(zip(le.classes_, np.bincount(y)))}')

Train tumor: (230, 60660)
Normal:      (58, 60660)
Classes: {'NON-TRU': np.int64(141), 'TRU': np.int64(89)}


## HVG from Bulk

In [ ]:
var_tumor  = tcga_train.var(axis=0)
var_normal = tcga_normal_t.var(axis=0)

common_genes = var_tumor.index.intersection(var_normal.index)
var_tumor  = var_tumor.loc[common_genes]
var_normal = var_normal.loc[common_genes]

eps = 1e-6
var_ratio = var_tumor / (var_normal + eps)

mean_tumor  = tcga_train.mean(axis=0).loc[common_genes]
mean_normal = tcga_normal_t.mean(axis=0).loc[common_genes]

keep = (mean_tumor > 1.0) & (mean_normal > 1.0)
print(f'Genes after expression filter: {keep.sum()}')

var_ratio = (var_tumor[keep] / (var_normal[keep] + 1e-6))

hvg_bulk_ens = set(var_ratio.nlargest(N_HVG).index)
print(f'Top-{N_HVG} by var_ratio: {len(hvg_bulk_ens)}')

Genes after expression filter: 14840
Top-3000 by var_ratio: 3000


In [ ]:
mg = mygene.MyGeneInfo()
top20_ens = var_ratio.nlargest(20).index.tolist()
res = mg.querymany(top20_ens, scopes='ensembl.gene', fields='symbol', species='human')
for r in res:
    print(r.get('query'), '→', r.get('symbol', '?'))

ModuleNotFoundError: No module named 'mygene'

In [ ]:
ens_list = list(hvg_bulk_ens)
res = mg.querymany(ens_list, scopes='ensembl.gene', fields='symbol', species='human')

ens_to_symbol = {
    r['query']: r['symbol']
    for r in res
    if not r.get('notfound') and 'symbol' in r
}
hvg_bulk_symbol = set(ens_to_symbol.values())
print(f'Converted to symbols: {len(hvg_bulk_symbol)} из {len(hvg_bulk_ens)}')

## 3. HVG from sc

In [ ]:
with open('../data/hvg_sc_lee_et_al_gene_list.txt') as f:
    hvg_sc = set(f.read().splitlines())

In [ ]:
list(hvg_sc)[:5], list(hvg_bulk_symbol)[:5]

## HVG cross among bulk and single-cell

In [ ]:

hvg_common = sorted(hvg_bulk_symbol & hvg_sc)
print(f'bulk ∩ sc: {len(hvg_common)}')

## Stratified Train/Test Split + KFold CV

In [ ]:

mg = mygene.MyGeneInfo()

ens = list(tcga_train.columns)

res = mg.querymany(
    ens,
    scopes='ensembl.gene',
    fields='symbol',
    species='human'
)

ens_to_symbol = {
    r['query']: r['symbol']
    for r in res
    if not r.get('notfound') and 'symbol' in r
}
tcga_train.columns = [ens_to_symbol.get(c, c) for c in tcga_train.columns]

In [ ]:
hvg_scvailable = [g for g in hvg_common if g in tcga_train.columns]
print(f'Genes in bulk matrix: {len(hvg_scvailable)}')

X_all = tcga_train[hvg_scvailable].values.astype(np.float32)
y_all = y

print(f'X_all: {X_all.shape},  y_all: {y_all.shape}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.3,
    stratify=y_all,
    random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]} samples')
print(f'Test:  {X_test.shape[0]} samples')
print('train ', dict(zip(le.classes_, np.bincount(y_train))))
print('test class distribution: ', dict(zip(le.classes_, np.bincount(y_test))))

In [ ]:
def make_xgb():
    return xgb.XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.3,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_scores   = []
all_y_true  = []
all_y_prob  = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    m = make_xgb()
    m.fit(X_train[tr_idx], y_train[tr_idx])
    preds = m.predict(X_train[val_idx])
    probs = m.predict_proba(X_train[val_idx])
    acc   = accuracy_score(y_train[val_idx], preds)
    cv_scores.append(acc)
    all_y_true.extend(y_train[val_idx])
    all_y_prob.extend(probs[:, 1])
    print(f'  Fold {fold}: accuracy={acc:.3f}')

print(f'\nCV mean accuracy: {np.mean(cv_scores):.3f} ± {np.std(cv_scores):.3f}')

all_y_true = np.array(all_y_true)
all_y_prob = np.array(all_y_prob)

macro_auc = roc_auc_score(all_y_true, all_y_prob)  
print(f'CV macro ROC-AUC: {macro_auc:.3f}')

In [ ]:
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(all_y_true, all_y_prob)          
auc = roc_auc_score(all_y_true, all_y_prob)

axes[0].plot(fpr, tpr, color='steelblue', label=f'TRU (AUC={auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].set_title('ROC-AUC (5-fold CV, train)')
axes[0].legend(fontsize=8)

y_pred_cv = (all_y_prob >= 0.5).astype(int)              
cm = confusion_matrix(all_y_true, y_pred_cv)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(
    ax=axes[1], colorbar=False, xticks_rotation=45
)
axes[1].set_title('Confusion Matrix (5-fold CV)')

plt.tight_layout()
plt.savefig('../figures/cv_results.png', dpi=150)
plt.show()

## Iterative SHAP Feature Elimination

In [ ]:
current_genes = list(hvg_scvailable)   # start from full cross
elimination_log = []                    
feature_log = []

print(f'Start: {len(current_genes)} genes')
print(f'Finish: {ELIM_MIN}–{ELIM_MAX} genes')
print(f'ШElimination step: {ELIM_STEP} гgenes/iter')
print('─' * 55)

iteration = 0
gene_to_idx = {g:i for i, g in enumerate(hvg_scvailable)}
while len(current_genes) > ELIM_MAX:
    iteration += 1
    n = len(current_genes)

    gene_idx = [gene_to_idx[g] for g in current_genes]
    X_tr_curr = pd.DataFrame(
        X_train[:, gene_idx],
        columns=current_genes
    )
    X_val_curr   = X_test[:, gene_idx]

    model_iter = make_xgb()
    model_iter.fit(X_tr_curr, y_train)

    skf_quick = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    acc_quick = cross_val_score(
        model_iter, X_tr_curr, y_train,
        cv=skf_quick, scoring='accuracy', n_jobs=-1
    ).mean()

    explainer = shap.TreeExplainer(model_iter)
    shap_values = explainer.shap_values(X_tr_curr)

    shap_importance = np.abs(shap_values).mean(axis=0)

    importance_series = pd.Series(shap_importance, index=current_genes)
    genes_to_drop = importance_series.nsmallest(ELIM_STEP).index.tolist()

    top_genes = importance_series.sort_values(ascending=False).head(20).index.tolist()
    bottom_genes = importance_series.sort_values().head(20).index.tolist()

    elimination_log.append({
        'iteration': iteration,
        'n_genes': n,
        'cv_acc': float(acc_quick),
        'mean_shap': float(importance_series.mean()),
        'std_shap': float(importance_series.std()),
        'dropped': genes_to_drop,
        'top20': top_genes
        })

    feature_log.append(
        pd.DataFrame({
            'gene': importance_series.index,
            'shap': importance_series.values,
            'iteration': iteration
        })
    )

    if iteration % 10 == 0 or n <= ELIM_MAX + 50:
        print(f'Iter {iteration:3d} | genes={n:4d} | CV acc={acc_quick:.3f}')

    current_genes = [g for g in current_genes if g not in genes_to_drop]

print(f'\nIteration done. Genes left: {len(current_genes)}')

In [ ]:
# final 5-fold CV
gene_idx_final = [hvg_scvailable.index(g) for g in current_genes]
X_tr_final     = X_train[:, gene_idx_final]
X_te_final     = X_test[:, gene_idx_final]

cv_final = cross_val_score(
    make_xgb(), X_tr_final, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='accuracy', n_jobs=-1
)
print(f'Final set: {len(current_genes)} генов')
print(f'5-fold CV accuracy: {cv_final.mean():.3f} ± {cv_final.std():.3f}')

In [ ]:
# obtaining only tumor-specific genes
with open('../data/genes_to_keep.txt') as f:
    genes_tumor_specific = set(line.strip() for line in f if line.strip())

current_genes = [g for g in current_genes if g in genes_tumor_specific]
print(f'Tumor-specific: {len(current_genes)} genes')

gene_idx_final = [hvg_scvailable.index(g) for g in current_genes]
X_tr_final = X_train[:, gene_idx_final]
X_te_final = X_test[:, gene_idx_final]

# final cv
cv_final = cross_val_score(
    make_xgb(), X_tr_final, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='accuracy', n_jobs=-1
)
print(f'CV accuracy after filter: {cv_final.mean():.3f} ± {cv_final.std():.3f}')

In [ ]:
# SHAP feature elimination accuracy curve
elim_df = pd.DataFrame(elimination_log)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(elim_df['n_genes'], elim_df['cv_acc'], lw=1.5, color='steelblue')
ax.axvline(len(current_genes), color='tomato', ls='--', label=f'Final genes: {len(current_genes)}')
ax.set_xlabel('Количество генов')
ax.set_ylabel('CV Accuracy (3-fold)')
ax.set_title('SHAP Feature Elimination Curve')
ax.invert_xaxis()
ax.legend()
plt.tight_layout()
plt.savefig('../figures/shap_elimination_curve.png', dpi=150)
plt.show()

## Final model

In [ ]:
model_final = make_xgb()
model_final.fit(X_tr_final, y_train)

y_pred_test = model_final.predict(X_te_final)
y_prob_test = model_final.predict_proba(X_te_final)

test_acc = accuracy_score(y_test, y_pred_test)
test_auc = roc_auc_score(y_test, y_prob_test[:, 1])

print(f'Test accuracy:      {test_acc:.3f}')
print(f'Test macro ROC-AUC: {test_auc:.3f}')
print()
print(classification_report(y_test, y_pred_test, target_names=le.classes_))


In [ ]:
explainer_final = shap.TreeExplainer(model_final)
shap_values_final = explainer_final.shap_values(X_te_final)

shap_importance_final = np.abs(shap_values_final).mean(axis=0)

shap_importance_final = pd.Series(
    shap_importance_final,
    index=current_genes
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
shap_importance_final.head(30).sort_values().plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('mean(|SHAP|)')
ax.set_title('Top 30 features by SHAP')
plt.tight_layout()
plt.savefig('../figures/shap_top30.png', dpi=150)
plt.show()
print(shap_importance_final.head(30).sort_values(ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y_test, y_prob_test[:, 1])
auc = roc_auc_score(y_test, y_prob_test[:, 1])
axes[0].plot(fpr, tpr, color='steelblue', label=f'AUC={auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].set_title('ROC-AUC (test set)')
axes[0].legend(fontsize=10)

ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_test),
    display_labels=le.classes_
).plot(ax=axes[1], colorbar=False, xticks_rotation=45)
axes[1].set_title('Confusion Matrix (test set)')

plt.tight_layout()
plt.savefig('../figures/test_results.png', dpi=150)
plt.show()

## Weighted features

In [ ]:
# Bossting weight for PI features

ifn_genes = ['MX1', 'MX2', 'IFI6', 'IFI16', 'BST2', 'AIM2', 'SLFN11']
ifn_in_model = [g for g in ifn_genes if g in current_genes]
ifn_idx = [current_genes.index(g) for g in ifn_in_model]

print(f'Boosting {len(ifn_in_model)} IFN-features: {ifn_in_model}')

feature_weights = np.ones(X_tr_final.shape[1])
for idx in ifn_idx:
    feature_weights[idx] = 5.0

model_boosted = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.3,
    eval_metric='mlogloss', random_state=RANDOM_STATE,
    n_jobs=-1,
    feature_weights=feature_weights
)

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
model_boosted.fit(X_tr_final, y_train, sample_weight=sample_weights)

y_pred_b = model_boosted.predict(X_te_final)
y_prob_b = model_boosted.predict_proba(X_te_final)

test_acc_b = accuracy_score(y_test, y_pred_b)
test_auc_b = roc_auc_score(label_binarize(y_test, classes=list(range(len(le.classes_)))), y_prob_b, average='macro')

print(f'Test acc: {test_acc_b:.3f}, AUC: {test_auc_b:.3f}')
print(classification_report(y_test, y_pred_b, target_names=le.classes_))

## Artefacts saving

In [ ]:
joblib.dump(model_final, '../models/xgb_luad_final_BIN.pkl')
joblib.dump(le,          '../models/label_encoder_BIN.pkl')
joblib.dump(model_boosted, '../models/xgb_luad_boosted.pkl')
np.save('../models/supplementary/hvg_bulk_BIN.npy',       np.array(sorted(hvg_bulk_symbol)))
np.save('../models/supplementary/hvg_sc_BIN.npy',         np.array(sorted(hvg_sc)))
np.save('../models/supplementary/hvg_common_BIN.npy',     np.array(hvg_common))
np.save('../models/supplementary/genes_final_BIN.npy',    np.array(current_genes))

shap_importance_final.to_csv('../models/supplementary/shap_importance_final_BIN.csv')
elim_df.to_csv('../models/supplementary/elimination_log_BIN.csv', index=False)

print('Saved:')
print('  xgb_luad_final_BIN.pkl        — final model')
print('  label_encoder_BIN.pkl         — label encoder')
print('  xgb_luad_boosted.pkl          — model with boosted features')
print('  hvg_bulk_BIN.npy              — HVG from bulk')
print('  hvg_sc_BIN.npy                — HVG from sc (pseudobulk)')
print('  hvg_common_BIN.npy            — primary cross between sc and bulk')
print('  genes_final_BIN.npy           — final feature set')
print('  shap_importance_final_BIN.csv — feature importance')
print('  elimination_log_BIN.csv       — elimination log')
print()
print(f'Total: {len(current_genes)} Genes | '
      f'Test acc={test_acc:.3f} | Test AUC={test_auc:.3f}')